In [1]:
#Q1. User Profile with Encapsulation
class User:
    def __init__(self, username, email, age):
        self._username = username        # private attribute
        self.set_email(email)            # validate on creation
        self.set_age(age)                # validate on creation

    # ─── EMAIL ───────────────────────────
    def get_email(self):
        return self._email

    def set_email(self, email):
        if "@" not in email or "." not in email:
            raise ValueError("Invalid email format")
        self._email = email

    # ─── AGE ─────────────────────────────
    def get_age(self):
        return self._age

    def set_age(self, age):
        if age < 18 or age > 120:
            raise ValueError("Age must be between 18 and 120")
        self._age = age

    # ─── USERNAME ─────────────────────────
    def get_username(self):
        return self._username


# ─── RUNNING THE CODE ─────────────────────
user = User("alice", "alice@mail.com", 25)

# Test invalid email
try:
    user.set_email("invalid")
except ValueError as e:
    print(f"ValueError: {e}")

# Test invalid age
try:
    user.set_age(150)
except ValueError as e:
    print(f"ValueError: {e}")

# Print original values (unchanged)
print(user.get_email())
print(user.get_age())

ValueError: Invalid email format
ValueError: Age must be between 18 and 120
alice@mail.com
25


In [2]:
#Q2. Inheritance — Admin and Customer Users
# ─── BASE CLASS ───────────────────────────────────────
class User:
    def __init__(self, username, role):
        self.username = username
        self.role     = role

    def display_profile(self):
        print(f"User: {self.username} | Role: {self.role}")


# ─── SUBCLASS 1 ───────────────────────────────────────
class AdminUser(User):
    def __init__(self, username, permissions):
        super().__init__(username, role="admin")   # calls User.__init__
        self.permissions = permissions             # extra attribute

    def display_profile(self):                     # override ✅
        perms = ", ".join(self.permissions)        # list → string
        print(f"Admin: {self.username} | Permissions: {perms}")


# ─── SUBCLASS 2 ───────────────────────────────────────
class CustomerUser(User):
    def __init__(self, username, orders_count):
        super().__init__(username, role="customer") # calls User.__init__
        self.orders_count = orders_count            # extra attribute

    def display_profile(self):                      # override ✅
        print(f"Customer: {self.username} | Orders: {self.orders_count}")


# ─── RUNNING THE CODE ─────────────────────────────────
admin    = AdminUser("admin1", ["manage_users", "view_logs"])
customer = CustomerUser("cust1", 5)

admin.display_profile()
customer.display_profile()

Admin: admin1 | Permissions: manage_users, view_logs
Customer: cust1 | Orders: 5


In [3]:
#Q3. Composition — Order with Address and Payment 
# ─── ADDRESS CLASS ────────────────────────────────────
class Address:
    def __init__(self, city, zip_code):
        self.city     = city
        self.zip_code = zip_code

    def display(self):
        return f"{self.city} - {self.zip_code}"


# ─── PAYMENT CLASS ────────────────────────────────────
class PaymentInfo:
    def __init__(self, method, amount):
        self.method = method
        self.amount = amount

    def display(self):
        return f"{self.method}"


# ─── ORDER ITEM CLASS ─────────────────────────────────
class OrderItem:
    def __init__(self, name, qty, price):
        self.name  = name
        self.qty   = qty
        self.price = price

    def total(self):
        return self.qty * self.price          # qty * price

    def display(self):
        return f"{self.name} x{self.qty} = {self.total()}"


# ─── ORDER CLASS (COMPOSITION) ────────────────────────
class Order:
    def __init__(self, address, payment, items):
        self.address = address                # Address object
        self.payment = payment                # PaymentInfo object
        self.items   = items                  # list of OrderItems

    def order_summary(self):
        # shipping
        print(f"Shipping: {self.address.display()}")

        # items
        items_display = ", ".join([item.display() for item in self.items])
        print(f"Items: {items_display}")

        # total
        total = sum([item.total() for item in self.items])
        print(f"Total: {total}")

        # payment
        print(f"Payment: {self.payment.display()}")


# ─── RUNNING THE CODE ─────────────────────────────────
addr  = Address("Bangalore", "560001")
pay   = PaymentInfo("UPI", 1500)
items = [OrderItem("Book", 2, 500), OrderItem("Pen", 5, 100)]
order = Order(addr, pay, items)

order.order_summary()

Shipping: Bangalore - 560001
Items: Book x2 = 1000, Pen x5 = 500
Total: 1500
Payment: UPI


In [4]:
#Q4. SRP — Separate Validation, Storage, and Notification 
import json
import os

# ─── CLASS 1: VALIDATOR ───────────────────────────────
# Only responsible for validating user data
class UserValidator:
    def validate(self, data):
        if "username" not in data or not data["username"]:
            raise ValueError("Username is required")

        if "email" not in data or "@" not in data["email"] or "." not in data["email"]:
            raise ValueError("Invalid email format")

        print("Validation passed")


# ─── CLASS 2: STORAGE ─────────────────────────────────
# Only responsible for reading and writing to JSON file
class UserStorage:
    def __init__(self, filename="users.json"):
        self.filename = filename

    def save(self, data):
        # read existing users
        users = self._read()

        # add new user
        users.append(data)

        # write back to file
        with open(self.filename, "w") as f:
            json.dump(users, f, indent=4)

        print(f"User saved to {self.filename}")

    def _read(self):
        # if file exists read it
        if os.path.exists(self.filename):
            with open(self.filename, "r") as f:
                return json.load(f)
        # if file doesn't exist return empty list
        return []


# ─── CLASS 3: NOTIFIER ────────────────────────────────
# Only responsible for sending notifications
class UserNotifier:
    def send_welcome(self, email):
        print(f"Welcome email sent to {email}")


# ─── ORCHESTRATOR FUNCTION ────────────────────────────
# Combines all three classes together
def register_user(data):
    validator = UserValidator()
    storage   = UserStorage()
    notifier  = UserNotifier()

    # Step 1 — validate
    validator.validate(data)

    # Step 2 — save
    storage.save(data)

    # Step 3 — notify
    notifier.send_welcome(data["email"])


# ─── RUNNING THE CODE ─────────────────────────────────
data = {"username": "alice", "email": "alice@mail.com"}
register_user(data)

Validation passed
User saved to users.json
Welcome email sent to alice@mail.com


In [5]:
#Q5. OCP — Extensible Discount System 
from abc import ABC, abstractmethod

# ─── ABSTRACT BASE CLASS ──────────────────────────────
# defines the contract all discounts must follow
class Discount(ABC):
    @abstractmethod
    def apply(self, amount):
        pass                          # subclasses MUST implement this


# ─── DISCOUNT 1: NO DISCOUNT ──────────────────────────
class NoDiscount(Discount):
    def apply(self, amount):
        return amount                 # no change, return as is


# ─── DISCOUNT 2: PERCENTAGE DISCOUNT ─────────────────
class PercentageDiscount(Discount):
    def apply(self, amount):
        result = amount - (amount * 0.10)   # 10% off
        return max(0, result)               # minimum 0


# ─── DISCOUNT 3: FLAT DISCOUNT ────────────────────────
class FlatDiscount(Discount):
    def apply(self, amount):
        result = amount - 200               # Rs 200 off
        return max(0, result)               # minimum 0


# ─── DISCOUNT 4: BUY ONE GET ONE FREE ─────────────────
class BuyOneGetOneFree(Discount):
    def apply(self, amount):
        result = amount * 0.50              # 50% off
        return max(0, result)               # minimum 0


# ─── CALCULATE TOTAL ──────────────────────────────────
# never changes, even when new discounts are added
def calculate_total(amount, discount):
    return discount.apply(amount)           # just calls apply()


# ─── RUNNING THE CODE ─────────────────────────────────
print(calculate_total(1000, NoDiscount()))
print(calculate_total(1000, PercentageDiscount()))
print(calculate_total(1000, FlatDiscount()))
print(calculate_total(1000, BuyOneGetOneFree()))

1000
900.0
800
500.0


In [6]:
#Q6. LSP — Fix the Bird Hierarchy 
from abc import ABC, abstractmethod

# ─── BASE CLASS ───────────────────────────────────────
# every bird can move — that's all we guarantee
class Bird(ABC):
    @abstractmethod
    def move(self):
        pass


# ─── FLYING BIRD ──────────────────────────────────────
# only birds that CAN fly inherit this
class FlyingBird(Bird):
    @abstractmethod
    def fly(self):
        pass

    def move(self):
        self.fly()                    # move = fly for flying birds


# ─── SWIMMING BIRD ────────────────────────────────────
# only birds that CAN swim inherit this
class SwimmingBird(Bird):
    @abstractmethod
    def swim(self):
        pass

    def move(self):
        self.swim()                   # move = swim for swimming birds


# ─── SPARROW ──────────────────────────────────────────
class Sparrow(FlyingBird):            # IS-A FlyingBird ✅
    def fly(self):
        print("Sparrow flies")


# ─── EAGLE ────────────────────────────────────────────
class Eagle(FlyingBird):              # IS-A FlyingBird ✅
    def fly(self):
        print("Eagle flies")


# ─── PENGUIN ──────────────────────────────────────────
class Penguin(SwimmingBird):          # IS-A SwimmingBird ✅
    def swim(self):
        print("Penguin swims")


# ─── DUCK ─────────────────────────────────────────────
# Duck BOTH flies and swims
class Duck(FlyingBird, SwimmingBird): # inherits BOTH ✅
    def fly(self):
        print("Duck flies")

    def swim(self):
        print("Duck swims")

    def move(self):                   # overrides both move()
        print("Duck flies and swims") # custom behavior


# ─── RUNNING THE CODE ─────────────────────────────────
for bird in [Sparrow(), Eagle(), Penguin(), Duck()]:
    bird.move()

Sparrow flies
Eagle flies
Penguin swims
Duck flies and swims


In [8]:
import json
import os
from abc import ABC, abstractmethod

class UserRepository(ABC):
    @abstractmethod
    def save(self, user):
        pass

    @abstractmethod
    def find(self, username):
        pass

class InMemoryUserRepository(UserRepository):
    def __init__(self):
        self.users = {}

    def save(self, user):
        self.users[user["username"]] = user
        print(f"User saved in memory")

    def find(self, username):
        return self.users.get(username, None)

class JSONUserRepository(UserRepository):
    def __init__(self, filename="users.json"):
        self.filename = filename

    def save(self, user):
        users = self._read()
        users[user["username"]] = user
        with open(self.filename, "w") as f:
            json.dump(users, f, indent=4)
        print(f"User saved to {self.filename}")

    def find(self, username):
        users = self._read()
        return users.get(username, None)

    def _read(self):
        if os.path.exists(self.filename):
            with open(self.filename, "r") as f:
                data = json.load(f)
                if isinstance(data, list):   # ← handles old format ✅
                    return {}
                return data
        return {}                            # fresh start ✅

class UserService:
    def __init__(self, repository: UserRepository):
        self.repository = repository

    def register(self, user):
        self.repository.save(user)

    def get_user(self, username):
        return self.repository.find(username)

# ─── RUNNING THE CODE ─────────────────────────────────
print("─── IN MEMORY ───")
repo    = InMemoryUserRepository()
service = UserService(repo)
service.register({"username": "alice", "email": "a@b.com"})
print(service.get_user("alice"))

print("\n─── JSON FILE ───")
repo    = JSONUserRepository()
service = UserService(repo)
service.register({"username": "bob", "email": "b@c.com"})
print(service.get_user("bob"))

─── IN MEMORY ───
User saved in memory
{'username': 'alice', 'email': 'a@b.com'}

─── JSON FILE ───
User saved to users.json
{'username': 'bob', 'email': 'b@c.com'}


In [9]:
#Q8. Thread vs Sequential — IO Simulation 
import threading
import time
from concurrent.futures import ThreadPoolExecutor


# ─── FETCH FUNCTION ───────────────────────────────────
def fetch_data(source, delay):
    print(f"  [{source}] started")
    time.sleep(delay)                    # simulates API call
    print(f"  [{source}] finished")


# ─── SEQUENTIAL ───────────────────────────────────────
def run_sequential(sources):
    print("\n─── SEQUENTIAL ───")
    start = time.time()

    for source, delay in sources:        # one by one, waits each time
        fetch_data(source, delay)

    total = time.time() - start
    print(f"Sequential time: {total:.1f}s")


# ─── THREADED ─────────────────────────────────────────
def run_threaded(sources):
    print("\n─── THREADED ───")
    start = time.time()

    threads = []
    for source, delay in sources:
        t = threading.Thread(
            target=fetch_data,
            args=(source, delay)         # pass source and delay
        )
        threads.append(t)
        t.start()                        # start immediately

    for t in threads:
        t.join()                         # wait for all to finish

    total = time.time() - start
    print(f"Threaded time: {total:.1f}s")


# ─── THREAD POOL (ALTERNATIVE) ────────────────────────
def run_threadpool(sources):
    print("\n─── THREAD POOL ───")
    start = time.time()

    with ThreadPoolExecutor(max_workers=5) as executor:
        for source, delay in sources:
            executor.submit(fetch_data, source, delay)  # submit each task

    total = time.time() - start
    print(f"Thread pool time: {total:.1f}s")


# ─── RUNNING THE CODE ─────────────────────────────────
sources = [
    ("users",     2),
    ("orders",    3),
    ("products",  1),
    ("reviews",   2),
    ("inventory", 1)
]

run_sequential(sources)
run_threaded(sources)
run_threadpool(sources)


─── SEQUENTIAL ───
  [users] started
  [users] finished
  [orders] started
  [orders] finished
  [products] started
  [products] finished
  [reviews] started
  [reviews] finished
  [inventory] started
  [inventory] finished
Sequential time: 9.0s

─── THREADED ───
  [users] started
  [orders] started
  [products] started
  [reviews] started
  [inventory] started
  [products] finished  [inventory] finished

  [users] finished
  [reviews] finished
  [orders] finished
Threaded time: 3.0s

─── THREAD POOL ───
  [users] started
  [orders] started
  [products] started
  [reviews] started
  [inventory] started
  [products] finished
  [inventory] finished
  [users] finished  [reviews] finished

  [orders] finished
Thread pool time: 3.0s


In [11]:
# import asyncio
# import time


# # ─── FETCH FUNCTION ───────────────────────────────────
# async def fetch_data(source, delay):
#     print(f"  [{source}] started")
#     await asyncio.sleep(delay)           # non-blocking wait ✅
#     print(f"  [{source}] finished")


# # ─── SEQUENTIAL ───────────────────────────────────────
# async def run_sequential(sources):
#     print("\n─── SEQUENTIAL ───")
#     start = time.time()

#     for source, delay in sources:
#         await fetch_data(source, delay)  # waits each one fully

#     total = time.time() - start
#     print(f"Sequential time: {total:.1f}s")


# # ─── ASYNC CONCURRENT ─────────────────────────────────
# async def run_async(sources):
#     print("\n─── ASYNC (gather) ───")
#     start = time.time()

#     await asyncio.gather(                # runs all concurrently ✅
#         *[fetch_data(source, delay) for source, delay in sources]
#     )

#     total = time.time() - start
#     print(f"Async time: {total:.1f}s")


# # ─── ENTRY POINT ──────────────────────────────────────
# async def main():
#     sources = [
#         ("users",     2),
#         ("orders",    3),
#         ("products",  1),
#         ("reviews",   2),
#         ("inventory", 1)
#     ]

#     await run_sequential(sources)
#     await run_async(sources)

# asyncio.run(main())

In [12]:
#Q9. Race Condition — Shared Counter Fix 
import threading


# ─── VERSION 1: WITHOUT LOCK (RACE CONDITION) ─────────
def increment_unsafe(container):
    for _ in range(1000):
        container[0] += 1              # not thread safe ❌


def run_without_lock():
    print("\n─── WITHOUT LOCK ───")
    container = [0]                    # mutable container (list)
    threads   = []

    for _ in range(10):
        t = threading.Thread(
            target=increment_unsafe,
            args=(container,)
        )
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    print(f"Without lock: {container[0]} (expected 10000)")


# ─── VERSION 2: WITH LOCK (THREAD SAFE) ───────────────
def increment_safe(container, lock):
    for _ in range(1000):
        with lock:                     # only one thread at a time ✅
            container[0] += 1


def run_with_lock():
    print("\n─── WITH LOCK ───")
    container = [0]                    # mutable container (list)
    lock      = threading.Lock()       # one lock shared by all threads
    threads   = []

    for _ in range(10):
        t = threading.Thread(
            target=increment_safe,
            args=(container, lock)     # pass lock to each thread
        )
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    print(f"With lock: {container[0]} (expected 10000)")


# ─── VERSION 3: USING CLASS (CLEANER) ─────────────────
class Counter:
    def __init__(self):
        self.value = 0
        self.lock  = threading.Lock()  # lock lives inside class

    def increment(self):
        with self.lock:                # protected ✅
            self.value += 1

    def get(self):
        return self.value


def run_with_class():
    print("\n─── WITH CLASS ───")
    counter = Counter()
    threads = []

    for _ in range(10):
        t = threading.Thread(
            target=lambda: [counter.increment() for _ in range(1000)]
        )
        threads.append(t)
        t.start()

    for t in threads:
        t.join()

    print(f"With class: {counter.get()} (expected 10000)")


# ─── SHOW INCONSISTENCY — RUN 5 TIMES ─────────────────
def show_inconsistency():
    print("\n─── RACE CONDITION (5 runs) ───")
    for run in range(1, 6):
        container = [0]
        threads   = []

        for _ in range(10):
            t = threading.Thread(
                target=increment_unsafe,
                args=(container,)
            )
            threads.append(t)
            t.start()

        for t in threads:
            t.join()

        status = "✅" if container[0] == 10000 else "❌"
        print(f"  Run {run}: {container[0]} {status}")


# ─── RUNNING THE CODE ─────────────────────────────────
show_inconsistency()       # proves race condition is real
run_without_lock()         # unsafe version
run_with_lock()            # fixed with lock
run_with_class()           # fixed with class


─── RACE CONDITION (5 runs) ───
  Run 1: 10000 ✅
  Run 2: 10000 ✅
  Run 3: 10000 ✅
  Run 4: 10000 ✅
  Run 5: 10000 ✅

─── WITHOUT LOCK ───
Without lock: 10000 (expected 10000)

─── WITH LOCK ───
With lock: 10000 (expected 10000)

─── WITH CLASS ───
With class: 10000 (expected 10000)


In [ ]:
#Q10. Multiprocessing — CPU-bound Speedup 
import time
import multiprocessing


# ─── CPU BOUND FUNCTION ───────────────────────────────
def compute_squares(n):
    result = sum(i * i for i in range(1, n + 1))  # heavy computation
    print(f"  compute_squares({n:,}) = {result}")
    return result


# ─── SEQUENTIAL ───────────────────────────────────────
def run_sequential(values):
    print("\n─── SEQUENTIAL ───")
    start   = time.time()
    results = []

    for n in values:
        result = compute_squares(n)    # one by one, blocks each time
        results.append(result)

    total = time.time() - start
    print(f"Sequential time: {total:.2f}s")
    return results


# ─── MULTIPROCESSING ──────────────────────────────────
def run_multiprocessing(values):
    print("\n─── MULTIPROCESSING ───")
    start = time.time()

    with multiprocessing.Pool() as pool:      # auto detects CPU cores
        results = pool.map(compute_squares, values)  # splits across cores

    total = time.time() - start
    print(f"Multiprocessing time: {total:.2f}s")
    return results


# ─── COMPARE RESULTS ──────────────────────────────────
def compare(seq_results, mp_results):
    print("\n─── VERIFICATION ───")
    all_match = seq_results == mp_results
    print(f"Results match: {all_match} ✅" if all_match else "Results mismatch ❌")


# ─── ENTRY POINT ──────────────────────────────────────
if __name__ == "__main__":              # required for multiprocessing!
    values = [10_000_000, 20_000_000, 15_000_000, 25_000_000]

    seq_results = run_sequential(values)
    mp_results  = run_multiprocessing(values)

    compare(seq_results, mp_results)


─── SEQUENTIAL ───
  compute_squares(10,000,000) = 333333383333335000000
  compute_squares(20,000,000) = 2666666866666670000000
  compute_squares(15,000,000) = 1125000112500002500000
  compute_squares(25,000,000) = 5208333645833337500000
Sequential time: 8.16s

─── MULTIPROCESSING ───


In [2]:
# #Q11. Async IO — Concurrent API Simulation 
# import asyncio
# import time


# # ─── SYNC FETCH ───────────────────────────────────────
# def fetch_sync(url, delay):
#     start = time.time()
#     print(f"  [SYNC]  {url} started  at {start - sync_start:.1f}s")
#     time.sleep(delay)                         # blocks everything ❌
#     end = time.time()
#     print(f"  [SYNC]  {url} finished at {end - sync_start:.1f}s")
#     return f"{url} data"


# # ─── ASYNC FETCH ──────────────────────────────────────
# async def fetch_async(url, delay):
#     start = time.time()
#     print(f"  [ASYNC] {url} started  at {start - async_start:.1f}s")
#     await asyncio.sleep(delay)                # yields control ✅
#     end = time.time()
#     print(f"  [ASYNC] {url} finished at {end - async_start:.1f}s")
#     return f"{url} data"


# # ─── SYNC RUNNER ──────────────────────────────────────
# def run_sync(urls):
#     global sync_start
#     print("\n─── SYNC VERSION ───")
#     sync_start = time.time()
#     results    = []

#     for url, delay in urls:
#         result = fetch_sync(url, delay)       # waits fully each time
#         results.append(result)

#     total = time.time() - sync_start
#     print(f"\nSync results : {results}")
#     print(f"Sync time    : {total:.1f}s")
#     return results


# # ─── ASYNC RUNNER ─────────────────────────────────────
# async def run_async(urls):
#     global async_start
#     print("\n─── ASYNC VERSION ───")
#     async_start = time.time()

#     results = await asyncio.gather(           # all run concurrently ✅
#         *[fetch_async(url, delay) for url, delay in urls]
#     )

#     total = time.time() - async_start
#     print(f"\nAsync results: {list(results)}")
#     print(f"Async time   : {total:.1f}s")
#     return results


# # ─── MAIN ─────────────────────────────────────────────
# async def main():
#     urls = [
#         ("api/users",    2),
#         ("api/orders",   3),
#         ("api/products", 1),
#         ("api/reviews",  2)
#     ]

#     run_sync(urls)              # sync first
#     await run_async(urls)       # async second


# asyncio.run(main())

In [3]:
#Q12. Pydantic — User Schema with Nested Validation 
from pydantic import BaseModel, EmailStr, Field, field_validator
from typing import Optional


# ─── ADDRESS MODEL ────────────────────────────────────
class Address(BaseModel):
    street  : str
    city    : str
    zip_code: str = Field(min_length=6, max_length=6)  # exactly 6 chars

    @field_validator("zip_code")
    def zip_must_be_digits(cls, value):
        if not value.isdigit():                        # must be numbers only
            raise ValueError("zip_code must contain only digits")
        return value


# ─── USER CREATE ──────────────────────────────────────
class UserCreate(BaseModel):
    username: str
    email   : str
    password: str  = Field(min_length=8)               # min 8 chars
    age     : int  = Field(ge=18, le=120)              # 18 to 120
    address : Address                                   # nested model ✅

    @field_validator("email")
    def email_must_have_at(cls, value):
        if "@" not in value:
            raise ValueError("email must contain @")
        return value


# ─── USER RESPONSE ────────────────────────────────────
class UserResponse(BaseModel):
    username: str
    email   : str
    age     : int
    address : Address
    # password NOT here → never exposed ✅


# ─── RUNNING THE CODE ─────────────────────────────────
data = {
    "username": "alice",
    "email"   : "alice@mail.com",
    "password": "securepass",
    "age"     : 25,
    "address" : {
        "street"  : "MG Road",
        "city"    : "Bangalore",
        "zip_code": "560001"
    }
}

# create and validate
user = UserCreate(**data)

# serialize — model_dump includes all fields
print("\n─── model_dump() ───")
print(user.model_dump())

# response — password excluded
print("\n─── UserResponse ───")
response = UserResponse(**user.model_dump())
print(response)

# show individual fields
print("\n─── Individual Fields ───")
print(f"Username : {response.username}")
print(f"Email    : {response.email}")
print(f"Age      : {response.age}")
print(f"City     : {response.address.city}")
print(f"Zip      : {response.address.zip_code}")


─── model_dump() ───
{'username': 'alice', 'email': 'alice@mail.com', 'password': 'securepass', 'age': 25, 'address': {'street': 'MG Road', 'city': 'Bangalore', 'zip_code': '560001'}}

─── UserResponse ───
username='alice' email='alice@mail.com' age=25 address=Address(street='MG Road', city='Bangalore', zip_code='560001')

─── Individual Fields ───
Username : alice
Email    : alice@mail.com
Age      : 25
City     : Bangalore
Zip      : 560001


In [ ]:
#Q13. FastAPI — Health Check and CRUD Endpoints 
